# 8 — The model, its calibration, and D₀

The generative model is an Urn Model with Triggering in which the probability of
sampling the adjacent possible is **attenuated** above a finite vocabulary scale
D₀. That single ingredient is what produces the crossover; without it the process
reduces to the stationary UMT.

This notebook builds the simulator, runs the four production trajectories and
the two D₀ sweeps. It writes no figure and no table: everything it produces is an
intermediate, and **notebook 9** turns those into Figure 4, Figure S5, Table S7
and Table S8 in seconds.

> **This is the expensive notebook — about an hour.** Four production runs at
> T = 10^8 (~8 min in total), then two D₀ sweeps: the wide one, 14 values × 2
> seeds × 2 populations (~10 min), and the deep one, 24 values × 6 seeds
> (~46 min). All of it is a one-off cost: once `data_reduced/sim_*.npz`,
> `outputs/d0_calibration_crossover/` and `outputs/d0_calibration_refined/`
> exist, nothing here needs to run again.

In [ ]:
import os
import subprocess
import sys

REPO = os.path.abspath("..") if os.path.isdir(os.path.join("..", "src")) else os.path.abspath(".")
sys.path.insert(0, os.path.join(REPO, "src"))

import plotting as P


def run(*command, must_succeed=True):
    """Run one pipeline step and, unlike a `!` cell, STOP if it fails.

    An IPython `!` cell throws away the exit status: a step that dies leaves no
    output, no error and no trace, and `nbconvert --execute` still reports the
    notebook as successful. Two defects in this pipeline's history hid exactly
    there, so every step below goes through this instead.

    `must_succeed=False` is used only for the two `--check` diagnostics of
    notebook 1, which print a loud banner rather than stopping the run.
    """
    print(">>", " ".join(str(c) for c in command), flush=True)
    code = subprocess.run([str(c) for c in command], check=False).returncode
    if code and must_succeed:
        raise RuntimeError(f"step failed with exit code {code} - read the output "
                           f"above; nothing after this point is valid")
    if code:
        rule = "*" * 72
        print(rule)
        print(f"*** THIS CHECK FAILED (exit code {code}). Read the output above")
        print("*** before going on: whatever depends on this corpus is missing")
        print("*** or wrong, and so is anything computed from it.")
        print(rule, flush=True)
    return code


def py(script, *args, must_succeed=True):
    """`run` for one of this repository's own scripts."""
    return run(sys.executable, os.path.join(REPO, "src", script), *args,
               must_succeed=must_succeed)


%matplotlib inline
USETEX = P.setup_style()
print("repo:", REPO, "| LaTeX text rendering:", USETEX)

## 8.1 Build the simulator

The simulator is C. `sim/optimized/UMT_dynamic_fast_zstd.c` is the production
version: it keeps the stochastic rules, the fixed seed and the initial urn of the
archived time-dependent simulator, and changes only the representation — a
Fenwick tree for the urn weights, so draws and updates are O(log D) instead of
scans with global renormalisation, and a uint32 history so stream draws are O(1).
Trajectories are written as Zstandard-compressed `CLTRJ1`.

`sim/tests/` holds two regression scripts that check the fast version against the
archived one byte for byte on short runs. Run them if you have changed anything
in `sim/`.

In [ ]:
run("make", "-C", os.path.join(REPO, "sim"))

In [ ]:
# regression checks against the archived simulator (short, bounded runs)
run("bash", os.path.join(REPO, "sim", "tests", "verify_roundtrip.sh"))
run("bash", os.path.join(REPO, "sim", "tests", "verify_fast_urn_equivalence.sh"))

## 8.2 The production runs

Four trajectories, each T = 10^8 steps, each reduced to the *same* on-disk schema
that `build_reduced.py` produces from the corpus — so model and data are compared
by identical code rather than by two similar-looking pipelines. The trajectory
itself is an intermediate and is deleted after reduction.

| run | what it is |
| --- | --- |
| `sim_calibrated` | the model at the calibrated D₀ = 9,366 |
| `sim_stationary` | D₀ set far above any reachable vocabulary, i.e. attenuation off |
| `sim_umt_rho2` | the stationary UMT with ρ = 2 |
| `sim_umt_transient` | ρ = 2 with a large initial urn, the baseline's strongest case |

Setting D₀ beyond reach is what makes the baseline comparison clean: it is the
same binary, same seed, same T, same reductions and same estimators, with exactly
one ingredient switched off.

In [ ]:
py("build_reduced_sim.py", "--run", "--d0", "9366", "--T", "100000000", "--name", "sim_calibrated")

In [ ]:
py("build_reduced_sim.py", "--run", "--d0", "1000000000", "--T", "100000000", "--name", "sim_stationary")
py("build_reduced_sim.py", "--run", "--d0", "1000000000", "--T", "100000000", "--rho", "2", "--name", "sim_umt_rho2")

The transient row needs its initial urn chosen first. `umt_transient_n0.py`
selects n₀ by bisection on log n₀ so that the baseline reproduces the *measured*
crossover — that is, it is tuned by the same criterion applied to our own model,
so the comparison is fair rather than convenient.

In [ ]:
py("umt_transient_n0.py")

In [ ]:
# substitute the n0 the bisection selected above (12,952 on the paper's run)
py("build_reduced_sim.py", "--run", "--d0", "1000000000", "--T", "100000000", "--rho", "2", "--N0", "12952", "--name", "sim_umt_transient")

## 8.3 Calibrating D₀

For every (D₀, seed) the simulator runs once and its trajectory is streamed —
never held in memory — into two summaries: the vocabulary growth at the empirical
sampling times, and the rank-frequency vector of the prefix whose token count
matches the empirical one. A population is only ever compared up to the token
count it actually has, whatever T the simulation ran for.

The sweeps are the expensive step, and there are two of them. They answer
different questions and neither replaces the other.

The **wide** sweep spans D₀ from 200 to 26,000 on two seeds and records
`rstar_sim`, the crossover of each simulated curve. That column is what
`figure_SI5.py` needs to build the crossover criterion, and it is why the grid
has to reach low: for the learners that criterion selects D₀ = 275, which a grid
starting higher could never find. 14 values x 2 seeds x 2 populations = 56 runs,
about 20 minutes.

The **deep** sweep narrows onto each population's loss minimum and runs six
seeds instead of two. Two seeds are not enough to say how wide the minimum is —
they collapse the learners' competitive range onto a single grid point. 24
values x 6 seeds = 288 runs, about 50 minutes.

Worth being clear about what the second one buys, since it is the longer:
**not a sharper D₀**. The natives come out at 9,366 either way, with the same
[6,664–13,163] range. The loss varies by 0.002 across a factor of 1.7 in D₀
while the spread between seeds is five times larger, so the estimate is limited
by how flat the objective is, not by how finely it is sampled. The extra seeds
buy an honest interval, not another digit.

Both cells need `--config` and `--tag`, and neither is optional. Without a config
the script falls back to `DEFAULTS`, a third grid that lands one point away —
D₀ = 9,902 for the natives. Without a tag everything lands in
`outputs/d0_calibration/`, and the modules that read
`outputs/d0_calibration_crossover/` and `outputs/d0_calibration_refined/` stop
with a `FileNotFoundError`.

The dry run gets its own tag so its two smoke-test runs cannot be mistaken for
either sweep.

In [ ]:
py("calibrate_d0.py", "--config", os.path.join(REPO, "manifests", "d0_calibration.json"), "--tag", "dryrun", "--dry-run")

In [ ]:
py("calibrate_d0.py", "--config", os.path.join(REPO, "manifests", "d0_calibration.json"), "--tag", "crossover", "--T", "100000000", "--allow-long-run")

In [ ]:
py("calibrate_d0.py", "--config", os.path.join(REPO, "manifests", "d0_calibration_refined.json"), "--tag", "refined", "--T", "100000000", "--allow-long-run")

## What you have now

`data_reduced/sim_*.npz` holds the four reduced runs, and both sweeps are on
disk: `outputs/d0_calibration_crossover/` (read by `figure_SI5.py` and
`model_vs_data.py`) and `outputs/d0_calibration_refined/` (read by
`estimate_d0.py`). Notebook 9 needs both. Nothing else in the repository runs
the simulator, so this notebook does not need to be run again.

**Next:** `09_model_and_d0.ipynb`, which reads these artifacts and produces the
figures and tables in seconds.